# Turtle Breakout 14 — the frozen specification on US equities: S&P 500 constituents and index ETFs

Pre-registration: `discussions/Turtle_Breakout/016_claude_tb14_preregistration.md`. Code: `turtle_stocks.py`, `turtle_portfolio.py` 0.3.0; data: `tools/download_tb14_stocks.py` (Yahoo Finance, daily, adjusted).

**Frozen spec:** one account; TB 01 daily longs + TB 07 reverse short at each losing stop exit; no portfolio rule; risk per trade $f \times$ equity; ≤ 3× notional per position; **gross entry notional ≤ 2× equity** (Reg-T).
**SP500:** the thesis constituent list (2026 snapshot, survivorship-biased), primary $f = 0.05\%$. **INDEX:** SPY, QQQ, DIA, IWM, primary $f = 0.5\%$. 2000 → last session.
**Gates per set** (Bonferroni over 4, $\alpha = 0.0125$ one-sided): A growth of long + reverse > 0; B Calmar(long + reverse) − Calmar(long) > 0.

Environment: `TB14_DATA` (default `<parent of DERIBIT_DATA_ROOT>/Stocks/tb14/stocks_daily.parquet`; **downloaded from Yahoo Finance on the first run if missing**, needs `pip install yfinance`), `TB14_SYNTHETIC=1`.

In [1]:
import os, sys, json, gc, time, subprocess, tempfile, warnings, platform
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'data_engineering').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src')); load_dotenv(ROOT / '.env')
from data_engineering.modeling import turtle_breakout as tb, turtle_sizing as ts, turtle_reverse as tr, turtle_portfolio as tp, turtle_oos as to, turtle_stocks as tk

SYNTHETIC = os.getenv('TB14_SYNTHETIC', '0') == '1'
SP = tk.SParams(); OP = to.OParams()
P = replace(tb.Params(), min_bar_share=0.0, funding=False, fee=SP.side_cost - tb.Params().slip)
RP = replace(tr.RParams(), side_cost=SP.side_cost)
PP = replace(tp.PParams(), size_base='equity', gross_cap=SP.gross_cap, status='PREREGISTERED_TB016')
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='tb14-')) / '14'; OP = to.OParams(n_boot=500)
    rng = np.random.default_rng(5); ses = pd.bdate_range('2000-01-03', periods=3000); rows = []
    mkt = np.cumsum(rng.normal(0.0003, 0.01, len(ses)))
    for i, t in enumerate([f'S{i}' for i in range(14)] + list(tk.INDEX_ETFS)):
        c = 50 * np.exp(0.5 * mkt + np.cumsum(rng.normal(0.0002, 0.015, len(ses)))); o = c * np.exp(rng.normal(0, 0.004, len(ses)))
        rows.append(pd.DataFrame({'date': ses, 'ticker': t, 'open': o, 'high': np.maximum(o, c) * 1.006, 'low': np.minimum(o, c) * 0.994, 'close': c, 'volume': 1.0}))
    RAW = pd.concat(rows, ignore_index=True)
else:
    OUT = ROOT / 'notebooks' / 'Turtle_Breakout' / '14'
    root = Path(os.environ['DERIBIT_DATA_ROOT']).parent
    DATA_FILE = Path(os.environ.get('TB14_DATA', str(root / 'Stocks' / 'tb14' / 'stocks_daily.parquet')))
    if not DATA_FILE.exists():                       # first run: download from Yahoo Finance (tools/download_tb14_stocks.py)
        sys.path.insert(0, str(ROOT / 'tools'))
        import download_tb14_stocks as dl
        print('stocks data not found, downloading to', DATA_FILE.parent)
        DATA_FILE = dl.download(DATA_FILE.parent)
    RAW = pd.read_parquet(DATA_FILE)
OUT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy(); env['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + env.get('PYTHONPATH', '')
for t in ('test_turtle_stocks.py', 'test_turtle_portfolio.py'):
    chk = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', t], cwd=str(ROOT), env=env, capture_output=True, text=True)
    print(t, chk.stderr.strip().splitlines()[-1]); assert chk.returncode == 0
RAW['date'] = pd.to_datetime(RAW['date']).dt.tz_localize(None).dt.normalize()
RAW = RAW[RAW.date >= pd.Timestamp(SP.start)]
SES = pd.DatetimeIndex(sorted(RAW.loc[RAW.ticker == 'SPY', 'date'].unique()))
print('sessions', SES[0].date(), '->', SES[-1].date(), len(SES), '| tickers', RAW.ticker.nunique())
print(json.dumps(tk.registration(SP), indent=1, default=str))

stocks data not found, downloading to D:\MarketData\Stocks\tb14
507 tickers, 1999-10-01 -> 2026-10-26
  50/507: 50 tickers with data
  100/507: 50 tickers with data
  150/507: 50 tickers with data
  200/507: 50 tickers with data
  250/507: 50 tickers with data
  300/507: 50 tickers with data
  350/507: 50 tickers with data
  400/507: 50 tickers with data
  450/507: 50 tickers with data


$WBD: possibly delisted; no price data found  (1d 1999-10-01 -> 2026-10-26)

1 Failed download:
['WBD']: possibly delisted; no price data found  (1d 1999-10-01 -> 2026-10-26)


  500/507: 49 tickers with data
  507/507: 7 tickers with data
saved 2,998,822 rows, 506 tickers; missing: 1 -> D:\MarketData\Stocks\tb14\stocks_daily.parquet
test_turtle_stocks.py OK
test_turtle_portfolio.py OK
sessions 2000-01-03 -> 2026-10-23 6743 | tickers 506
{
 "start": "2000-01-03",
 "side_cost": 0.0005,
 "side_cost_high": 0.001,
 "borrow": 0.005,
 "borrow_high": 0.02,
 "gross_cap": 2.0,
 "f_sp500": [
  0.00025,
  0.0005,
  0.001
 ],
 "f_sp500_primary": 0.0005,
 "f_index": [
  0.0025,
  0.005,
  0.01
 ],
 "f_index_primary": 0.005,
 "warmup_sessions": 30,
 "status": "PREREGISTERED_TB016",
 "version": "TB14-0.1.0",
 "index_etfs": [
  "SPY",
  "QQQ",
  "DIA",
  "IWM"
 ],
 "universe": "thesis S&P 500 snapshot (igarpet/thesis f967975)",
 "frozen_spec": "one account, TB 01 longs + TB 07 reverse, R0, f x equity, 3x notional per position, gross <= 2x",
 "gates": {
  "A": "long_rev at primary f: mean daily log growth, block-bootstrap lower bound at alpha/4 > 0",
  "B": "Calmar(long_rev) 

## 1. Books per ticker (main costs; 10 bp sensitivity; 2% borrow sensitivity)

In [2]:
SETS = {'sp500': {}, 'index': {}}
T0 = time.time()
for kind, names in (('index', list(tk.INDEX_ETFS)), ('sp500', sorted(set(RAW.ticker) - set(tk.INDEX_ETFS)))):
    books = {c: {'long': {}, 'long_rev': {}} for c in ('main', 'cost_high', 'borrow_high')}; closes = {}; cov = []
    for t in names:
        d = RAW[RAW.ticker == t].set_index('date')[['open', 'high', 'low', 'close']].sort_index()
        d = d[~d.index.duplicated()]
        if len(d) < SP.warmup_sessions + 60:
            cov.append({'ticker': t, 'status': 'too short', 'sessions': len(d)}); continue
        b = tk.pseudo_bars(d, SES)
        D = tb.prepare_asset(t, b, None, P)
        start = b.index[0] + pd.Timedelta(days=SP.warmup_sessions)
        L0 = ts.trade_list(D, P, start, RP, with_reverse=False); L1 = ts.trade_list(D, P, start, RP, with_reverse=True)
        for case, cost, borrow in (('main', SP.side_cost, SP.borrow), ('cost_high', SP.side_cost_high, SP.borrow), ('borrow_high', SP.side_cost, SP.borrow_high)):
            l0, l1 = L0.assign(cost=cost), L1.assign(cost=cost)
            books[case]['long'][t] = tp.make_book(D, l0, P)
            books[case]['long_rev'][t] = tk.apply_borrow(tp.make_book(D, l1, P), borrow, SES)
        closes[t] = d.close
        bm = books['main']
        cov.append({'ticker': t, 'status': 'ok', 'first': d.index[0].date(), 'last': d.index[-1].date(), 'sessions': len(d),
                    'longs': len(L0), 'reverses': int((L1.kind == 'reverse').sum()),
                    'mean_R_long': float(bm['long'][t].trades.R.mean()) if len(L0) else np.nan,
                    'mean_R_reverse': float(bm['long_rev'][t].trades.loc[bm['long_rev'][t].trades.kind == 'reverse', 'R'].mean()) if len(L1) > len(L0) else np.nan})
        del D, b; gc.collect()
    SETS[kind] = {'books': books, 'closes': closes, 'cov': pd.DataFrame(cov)}
    ok = SETS[kind]['cov'].query("status == 'ok'")
    print(f"{kind}: {len(ok)} tickers, {int(ok.longs.sum())} longs, {int(ok.reverses.sum())} reverses, "
          f"mean R long {ok.mean_R_long.mean():.3f}, reverse {ok.mean_R_reverse.mean():.3f} ({time.time() - T0:.0f}s)")

index: 4 tickers, 687 longs, 363 reverses, mean R long 0.204, reverse -0.253 (6s)


TypeError: Cannot convert tz-naive timestamps, use tz_localize to localize

## 2. One account, frozen spec

In [ ]:
def run(books, f_, t0):
    o = tp.simulate(books, 'R0', replace(PP, f=f_), t0)
    D = tk.real_daily(tp.daily_curve(o['equity'], PP.start_equity), SES)
    return D, {**tk.metrics(D, PP.start_equity), 'max_open_risk': o['max_open_risk'], 'mean_open_risk': o['mean_open_risk'],
               'max_gross_lev': o['max_gross_lev'], **o['stats']}
FS = {'sp500': SP.f_sp500, 'index': SP.f_index}; FP = {'sp500': SP.f_sp500_primary, 'index': SP.f_index_primary}
RES, CUR = [], {}
for kind in ('sp500', 'index'):
    S = SETS[kind]; S['t0'] = min(b.trades.t_in.min() for b in S['books']['main']['long'].values()) - pd.Timedelta(minutes=5)
    for f_ in FS[kind]:
        for v in ('long', 'long_rev'):
            D, m = run(S['books']['main'][v], f_, S['t0']); CUR[(kind, 'main', f_, v)] = D
            RES.append({'set': kind, 'case': 'main', 'f': f_, 'variant': v, **m})
    for case in ('cost_high', 'borrow_high'):
        for v in ('long', 'long_rev'):
            if case == 'borrow_high' and v == 'long':
                continue
            D, m = run(S['books'][case][v], FP[kind], S['t0']); CUR[(kind, case, FP[kind], v)] = D
            RES.append({'set': kind, 'case': case, 'f': FP[kind], 'variant': v, **m})
    print(kind, 'done')
RES = pd.DataFrame(RES)
display(RES[['set', 'case', 'f', 'variant', 'cagr', 'max_dd', 'calmar', 'sharpe', 'vol', 'ruined', 'max_open_risk', 'mean_open_risk', 'max_gross_lev', 'entries', 'skipped', 'scaled']].round(4))

## 3. Decision

In [ ]:
DEC = {}
for j, kind in enumerate(('sp500', 'index')):
    a = CUR[(kind, 'main', FP[kind], 'long_rev')]; b = CUR[(kind, 'main', FP[kind], 'long')]
    A = to.gate_growth(a, PP.start_equity, OP, OP.seed + j, ann=tk.ANN)
    B = to.gate_calmar(a, b, PP.start_equity, OP, OP.seed + 10 + j, ann=tk.ANN)
    DEC[kind] = {'f': FP[kind], 'A_growth': A, 'B_reversal': B, 'verdict': to.verdict(A['pass'], B['pass'])}
print(json.dumps(DEC, indent=1, default=float))

## 4. Reported: benchmarks, matched exposure, sub-periods, trades

In [ ]:
rows = []
spy = RAW[RAW.ticker == 'SPY'].set_index('date').close.sort_index()
for kind in ('sp500', 'index'):
    S = SETS[kind]; first = CUR[(kind, 'main', FP[kind], 'long')].index[0]
    lr = tk.equal_weight_hold(S['closes']); lr = lr[lr.index >= first]
    D = PP.start_equity * np.exp(lr.cumsum()); CUR[(kind, 'hold')] = D
    rows.append({'set': kind, 'benchmark': 'equal-weight daily-rebalanced buy and hold of the set', **tk.metrics(D, PP.start_equity)})
    s = spy[spy.index >= first]; Ds = PP.start_equity * s / s.iloc[0]; CUR[(kind, 'spy')] = Ds
    rows.append({'set': kind, 'benchmark': 'SPY buy and hold', **tk.metrics(Ds, PP.start_equity)})
    e = RES[(RES.set == kind) & (RES.case == 'main') & (RES.f == FP[kind])].set_index('variant')
    fm = FP[kind] * e.loc['long_rev', 'mean_open_risk'] / e.loc['long', 'mean_open_risk']
    D2, m2 = run(S['books']['main']['long'], fm, S['t0']); CUR[(kind, 'long_matched')] = D2
    rows.append({'set': kind, 'benchmark': f'long only at f = {fm:.4%} (exposure of long + reverse)', **m2,
                 'calmar_diff_long_rev_minus_this': to.gate_calmar(CUR[(kind, 'main', FP[kind], 'long_rev')], D2, PP.start_equity, OP, OP.seed + 20, ann=tk.ANN)['calmar_diff']})
BENCH = pd.DataFrame(rows); display(BENCH.round(3))

SUB = []
for kind in ('sp500', 'index'):
    for lab, (a, b) in tk.period_slices().items():
        for key in ((kind, 'main', FP[kind], 'long'), (kind, 'main', FP[kind], 'long_rev'), (kind, 'hold'), (kind, 'spy')):
            c = CUR[key]; c = c[(c.index >= a) & (c.index <= b)]
            if len(c) > 50:
                SUB.append({'set': kind, 'period': lab, 'series': key[-1], **tk.metrics(c / c.iloc[0] * PP.start_equity, PP.start_equity)})
SUB = pd.DataFrame(SUB); display(SUB.pivot_table(index=['set', 'period'], columns='series', values=['cagr', 'max_dd']).round(3))

TR = []
for kind in ('sp500', 'index'):
    T = pd.concat([b.trades.assign(ticker=t) for t, b in SETS[kind]['books']['main']['long_rev'].items()])
    for k_, g in T.groupby('kind'):
        TR.append({'set': kind, 'kind': k_, 'trades': len(g), 'mean_R': g.R.mean(), 'median_R': g.R.median(), 'win_rate': (g.R > 0).mean(),
                   'share_R_ge_3': (g.R >= 3).mean(), 'per_year': len(g) / (len(SES) / 252)})
TR = pd.DataFrame(TR); display(TR.round(3))

## 5. Figures

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
for j, kind in enumerate(('sp500', 'index')):
    for key, col, ls, lab in (((kind, 'main', FP[kind], 'long'), '#1f5aa6', '-', 'long'), ((kind, 'main', FP[kind], 'long_rev'), '#b33f40', '-', 'long + reverse'),
                              ((kind, 'long_matched'), '#1f5aa6', ':', 'long, matched exposure'), ((kind, 'hold'), '0.5', '--', 'equal-weight hold'), ((kind, 'spy'), '#3a7d44', '--', 'SPY')):
        c = CUR[key]
        axes[0, j].plot(c.index, c / PP.start_equity, color=col, ls=ls, lw=1.2, label=lab)
        axes[1, j].plot(c.index, (c / c.cummax() - 1) * 100, color=col, ls=ls, lw=0.9, label=lab)
    axes[0, j].set_yscale('log'); axes[0, j].set_title(f'{kind.upper()}: equity (f = {FP[kind]:.2%})', loc='left', fontsize=10)
    axes[1, j].set_title(f'{kind.upper()}: drawdown %', loc='left', fontsize=10)
    for a in axes[:, j]:
        a.grid(alpha=0.25); a.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

## 6. Save

In [ ]:
RES.to_csv(OUT / '14_results.csv', index=False); BENCH.to_csv(OUT / '14_benchmarks.csv', index=False); SUB.to_csv(OUT / '14_subperiods.csv', index=False)
TR.to_csv(OUT / '14_trades_summary.csv', index=False)
for kind in ('sp500', 'index'):
    SETS[kind]['cov'].to_csv(OUT / f'14_coverage_{kind}.csv', index=False)
pd.DataFrame({'|'.join(map(str, k)): c for k, c in CUR.items()}).to_csv(OUT / '14_curves.csv.gz', compression='gzip')
(OUT / '14_decision.json').write_text(json.dumps({'decision': DEC, 'registration': tk.registration(SP), 'synthetic': SYNTHETIC}, indent=1, default=str))
git = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=str(ROOT), capture_output=True, text=True).stdout.strip()
(OUT / '14_manifest.json').write_text(json.dumps({'version': tk.VERSION, 'portfolio_version': tp.VERSION, 'git': git, 'python': platform.python_version(),
    'created_utc': pd.Timestamp.now('UTC').isoformat(), 'sessions': [str(SES[0].date()), str(SES[-1].date())], 'synthetic': SYNTHETIC}, indent=1))
print('saved to', OUT); print({k: v['verdict'] for k, v in DEC.items()})